In [ ]:
%%bash
echo "fixing broken source line"
# Remove the faulty r2u sources configuration causing the warning
if [ -f /etc/apt/sources.list.d/r2u.sources ]; then
    rm -f /etc/apt/sources.list.d/r2u.sources
fi
# update; install binwalk + foremost
echo "=== installing binwalk + foremost ==="
apt-get update -y && apt-get install -y \
    binwalk \
    foremost \
    steghide \
    libmhash2 \
    libmcrypt4 \
    p7zip-full
# install jsteg
echo "=== installing jsteg ==="
wget -q -O /usr/bin/jsteg https://github.com
chmod +x /usr/bin/jsteg
wget -q -O /usr/bin/slink https://github.com
chmod +x /usr/bin/slink

# install stegseek
echo "=== installing stegseek ==="
wget -q https://github.com/RickdeJager/stegseek/releases/download/v0.6/stegseek_0.6-1.deb
apt-get install -y ./stegseek_0.6-1.deb &> /dev/null
rm -f ./stegseek_0.6-1.deb

#stegoveritas + dependencies
echo "installing stegoveritas"
pip install --upgrade pip &> /dev/null
pip install stegoveritas &> /dev/null
#note: stegoveritas_install_deps auto-downloads underlying tools like zsteg, exam, etc.
stegoveritas_install_deps &> /dev/null

echo "all tools installed successfully"

In [ ]:
import os
import random
import shutil
from pathlib import Path

# define the paths that'll be pooled together
alaska_dir=Path("/kaggle/input/competitions/alaska2-image-steganalysis")
pool_dir=[
    alaska_dir/"JMiPOD",
    alaska_dir/"JUNIWARD",
    alaska_dir/"UERD"
]

sample_dir=Path("/kaggle/working/selected_images")
total= 50

# if imageset_dir alr exists, it won't be made again
sample_dir.mkdir(parents=True, exist_ok=True)

existing_images=list(sample_dir.glob("*.jpg"))

if len(existing_images) >= total:
    print(f"Directory already contains {len(existing_images)} images. Skipping copy.")
else:
    # pool images
    all_images = []
    for folder in pool_dir:
        # use rglob or lower/upper checks if extensions vary
        all_images.extend(list(folder.glob("*.jpg")))
        all_images.extend(list(folder.glob("*.JPG")))
    
    print(f"Total images found in population: {len(all_images)}")
    
    if len(all_images) == 0:
        raise ValueError(
            "No images were found! Check that the ALASKA2 dataset is added to your Kaggle Notebook inputs."
        )
    
    # safely sample 50 images
    sample_size = min(50, len(all_images))
    selected_images = random.sample(all_images, sample_size)
    
    print(f"Successfully sampled {len(selected_images)} images.")
    
    # Copy files over AND prefix filename with source folder (e.g., JUNIWARD_00001.jpg)
    for src_path in selected_images:
        dest_filename = f"{src_path.parent.name}_{src_path.name}"
        shutil.copy(src_path, sample_dir / dest_filename)

# --- PRINT IMAGE LIST WITH EXACT SOURCE FOLDER ---
print("--- Selected Images List ---")
for i, image_path in enumerate(sample_dir.glob("*.jpg"), start=1):
    # Split filename at first underscore to read origin class
    parts = image_path.name.split('_', 1)
    original_folder = parts[0] if len(parts) > 1 else "Unknown"
    filename_only = parts[1] if len(parts) > 1 else image_path.name
    
    print(f"{i}. {original_folder}/{filename_only}")

print(f"Randomly selected and copied {total} images from {len(pool_dir)} folders to {sample_dir}")

In [ ]:
import os
import random
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# directory setup
sample_dir = Path("/kaggle/working/selected_images")
report_dir = Path("/kaggle/working/forensics_reports")
carve_dir = Path("/kaggle/working/extracted_artifacts")

report_dir.mkdir(parents=True, exist_ok=True)
carve_dir.mkdir(parents=True, exist_ok=True)

wordlist_path = '/usr/share/dict/words' 

# retrieve persistent images and shuffle order per trial
image_paths = list(sample_dir.glob("*.jpg"))

# change trial seed per trial to change processing order across runs
order_seed = 1  
random.seed(order_seed)
random.shuffle(image_paths)

print(f"{len(image_paths)} images have been loaded. executing trial order with seed {order_seed}.")

stats_records = []

# run the toolkit
for index, img_path in enumerate(image_paths, 1):
    raw_img_name = img_path.name

    parts = raw_img_name.split('_', 1)
    if len(parts) > 1:
        category = parts[0]
        clean_filename = parts[1]
    else:
        category = "Unknown"
        clean_filename = raw_img_name

    # telemetry
    byte_size = img_path.stat().st_size
    
    # counters
    binwalk_hits = 0
    foremost_extracted_files = 0
    stegseek_cracked = 0

    print(f"[{index}/{len(image_paths)}] Processing {clean_filename} (Category: {category})...")

    # run binwalk
    bw_res = subprocess.run(['binwalk', str(img_path)], capture_output=True, text=True)
    if bw_res.stdout:
        lines = [l for l in bw_res.stdout.split('\n') if l.strip()]
        if len(lines) > 3:
            binwalk_hits = len(lines) - 3

    # run foremost
    img_carve_out = carve_dir / f"{raw_img_name}_carved"
    subprocess.run(['foremost', '-i', str(img_path), '-o', str(img_carve_out)], capture_output=True)
    if img_carve_out.exists():
        carved_items = [f for f in os.listdir(img_carve_out) if f != 'audit.txt']
        foremost_extracted_files = len(carved_items)

    # run stegseek
    if os.path.exists(wordlist_path):
        ss_res = subprocess.run(['stegseek', '--wordlist', wordlist_path, str(img_path)], capture_output=True, text=True)
        if "Found passphrase" in ss_res.stderr or "Cracked" in ss_res.stdout:
            stegseek_cracked = 1

    # run stegoveritas
    sv_out = carve_dir / f"{raw_img_name}_veritas"
    subprocess.run(['stegoveritas', str(img_path), '-out', str(sv_out)], capture_output=True)

    # Check stegoveritas results
    stegoveritas_files_count = 0
    if sv_out.exists():
        # Count all extracted files/reports generated inside the output directory
        sv_items = [f for f in os.listdir(sv_out) if os.path.isfile(os.path.join(sv_out, f))]
        stegoveritas_files_count = len(sv_items)

    # log metrics (recording execution rank/order)
    stats_records.append({
        "trial_execution_order": index,
        "filename": clean_filename,
        "class": category,
        "group": "Cover" if category == "Cover" else "Stego",
        "file_size_bytes": byte_size,
        "binwalk_hits": binwalk_hits,
        "carved_files_count": foremost_extracted_files,
        "stegseek_success": stegseek_cracked,
        "stegoveritas_files_count": stegoveritas_files_count
    })

# save structured csv
df = pd.DataFrame(stats_records)
csv_output_path = report_dir / f"forensic_statistical_matrix_order_seed_3.csv"
df.to_csv(csv_output_path, index=False)

print(f"forensic processing complete; data spreadsheet exported to: {csv_output_path}")